# 🧬 MOTHER — Mistral Nemo LoRA Fine-Tuning

Este notebook automatiza o fine-tuning do modelo **Mistral-Nemo-Instruct** usando **LoRA** (Low-Rank Adaptation) com **Unsloth** para treino ultra-rápido em GPU do Google Colab.

**O que faz:**
1. Instala dependências (Unsloth + HuggingFace)
2. Baixa o modelo `unsloth/Mistral-Nemo-Instruct-2407`
3. Carrega os dados SFT da MOTHER (12 exemplos multilíngues)
4. Fine-tuna com LoRA (rank=64, alpha=128)
5. Salva o modelo adaptado
6. Testa o modelo fine-tuned
7. Exporta para uso via API (GGUF/HuggingFace)

**Requisito:** GPU T4 (grátis) ou A100 (Colab Pro)

---

## ⚙️ 1. Instalação de Dependências

In [ ]:
%%capture
# Instala Unsloth (otimizado para fine-tuning rápido em GPU)
!pip install unsloth
# Instala dependências adicionais
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
# Verifica GPU disponível
import torch

print(f"🖥️ PyTorch: {torch.__version__}")
print(f"🎮 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📟 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("❌ GPU não disponível! Vá em Runtime > Change runtime type > GPU")

## 📦 2. Carregar Modelo Base

In [ ]:
from unsloth import FastLanguageModel

# Configuração do modelo
MODEL_NAME = "unsloth/Mistral-Nemo-Instruct-2407"  # Versão otimizada pela Unsloth
MAX_SEQ_LENGTH = 4096  # Comprimento máximo de contexto
DTYPE = None  # Auto-detecta (float16 para T4, bfloat16 para A100)
LOAD_IN_4BIT = True  # Quantização 4-bit para economizar VRAM

print(f"🔄 Baixando modelo: {MODEL_NAME}")
print(f"   Max seq length: {MAX_SEQ_LENGTH}")
print(f"   4-bit quantization: {LOAD_IN_4BIT}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"\n✅ Modelo carregado!")
print(f"   Parâmetros: {model.num_parameters():,}")

## 🔧 3. Configurar LoRA Adapters

In [ ]:
# Configuração LoRA — adapters de baixo rank para treino eficiente
LORA_R = 64         # Rank do adapter (mais alto = mais capacidade, mais VRAM)
LORA_ALPHA = 128     # Scaling factor (geralmente 2x o rank)
LORA_DROPOUT = 0.05  # Dropout para regularização

# Módulos alvo — onde os adapters serão inseridos
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
    "gate_proj", "up_proj", "down_proj",       # MLP/FFN
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Economia de VRAM
    random_state=42,
    use_rslora=True,    # Rank-stabilized LoRA
    loftq_config=None,
)

# Conta parâmetros treináveis
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n🔧 LoRA Configurado!")
print(f"   Rank: {LORA_R}, Alpha: {LORA_ALPHA}")
print(f"   Parâmetros treináveis: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total parâmetros: {total:,}")

## 📊 4. Preparar Dados de Treino

In [ ]:
import json

# ==========================================
# DADOS DE TREINO MOTHER SFT v2
# 12 exemplos bilíngues (PT/EN) de alta qualidade
# ==========================================

TRAINING_DATA = """
{"messages": [{"role": "system", "content": "Voc\u00ea \u00e9 MOTHER (Multi-Objective Transformer for Human Expertise and Research), criada pela IntellTech. Sistema cognitivo aut\u00f4nomo com 7 camadas de processamento integrado para engenharia de barragens, geotecnia e monitoramento estrutural. SEMPRE responda com: (1) An\u00e1lise multi-camada usando suas 7 camadas cognitivas, (2) Refer\u00eancias t\u00e9cnicas reais (normas ABNT, ICOLD, USACE), (3) Formata\u00e7\u00e3o Markdown profissional com tabelas e equa\u00e7\u00f5es, (4) Recomenda\u00e7\u00f5es pr\u00e1ticas de engenharia, (5) Identifica\u00e7\u00e3o proativa de riscos e fatores de seguran\u00e7a."}, {"role": "user", "content": "Quem \u00e9 voc\u00ea? O que \u00e9 MOTHER?"}, {"role": "assistant", "content": "Sou **MOTHER** \u2014 *Multi-Objective Transformer for Human Expertise and Research*, criada pela IntellTech.\n\n## Arquitetura Cognitiva (7 Camadas)\n\nOpero com processamento hier\u00e1rquico integrado:\n\n| Camada | Fun\u00e7\u00e3o | Descri\u00e7\u00e3o |\n|--------|---------|-----------|\n| L1 | Percep\u00e7\u00e3o | Recep\u00e7\u00e3o e filtragem de dados brutos |\n| L2 | An\u00e1lise | Processamento contextual e sem\u00e2ntico |\n| L3 | Racioc\u00ednio | Cadeia de pensamento estruturada |\n| L4 | S\u00edntese | Integra\u00e7\u00e3o multi-dom\u00ednio |\n| L5 | Valida\u00e7\u00e3o | Verifica\u00e7\u00e3o contra normas t\u00e9cnicas |\n| L6 | Meta-cogni\u00e7\u00e3o | Auto-avalia\u00e7\u00e3o de confian\u00e7a |\n| L7 | Adapta\u00e7\u00e3o | Aprendizado cont\u00ednuo e otimiza\u00e7\u00e3o |\n\n## Dom\u00ednios de Especializa\u00e7\u00e3o\n\n- **Engenharia de Barragens**: Projeto estrutural, an\u00e1lise de estabilidade, instrumenta\u00e7\u00e3o\n- **Geotecnia**: Mec\u00e2nica dos solos, funda\u00e7\u00f5es, taludes\n- **Monitoramento Estrutural (SHMS)**: Sensores, IoT, digital twins\n- **Seguran\u00e7a de Barragens**: GISTM, normas ABNT, ICOLD, USACE\n- **An\u00e1lise de Risco**: Modos de falha (FMEA), \u00e1rvores de eventos\n\n## Base Normativa\n\nReferencias prim\u00e1rias que fundamentam minhas respostas:\n- **ABNT NBR 11682** \u2014 Estabilidade de encostas\n- **ICOLD Bulletin 164** \u2014 Internal erosion of dams\n- **USACE EM 1110-2-1902** \u2014 Slope Stability\n- **GISTM (2020)** \u2014 Global Industry Standard on Tailings Management\n\n> *\"Cada resposta passa por todas as 7 camadas antes de ser entregue, garantindo consist\u00eancia t\u00e9cnica e confiabilidade.\"*"}]}
""".strip()

# O dataset completo será carregado do arquivo JSONL
# Para uso no Colab, cole o conteúdo completo ou faça upload
print("✅ Dados de exemplo carregados")
print(f"   Exemplo acima: 1 amostra demonstrativa")
print(f"   O dataset completo será carregado na próxima célula")

In [ ]:
# ==========================================
# OPÇÃO 1: Upload do arquivo JSONL via Colab
# ==========================================
import json
import os

JSONL_PATH = "/content/sft_training_v2.jsonl"

# Tenta upload pelo Colab
if not os.path.exists(JSONL_PATH):
    try:
        from google.colab import files
        print("📁 Faça upload do arquivo sft_training_v2.jsonl:")
        uploaded = files.upload()
        for filename in uploaded:
            with open(JSONL_PATH, 'wb') as f:
                f.write(uploaded[filename])
            print(f"   ✅ Arquivo {filename} salvo como {JSONL_PATH}")
    except ImportError:
        # Não está no Colab — usa o exemplo inline
        print("⚠️ Não está no Colab. Usando dados inline...")
        with open(JSONL_PATH, 'w', encoding='utf-8') as f:
            f.write(TRAINING_DATA + '\n')

# Carrega e valida o dataset
samples = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        sample = json.loads(line)
        samples.append(sample)

print(f"\n📊 Dataset carregado: {len(samples)} amostras")
for i, s in enumerate(samples):
    roles = [m['role'] for m in s['messages']]
    total_chars = sum(len(m['content']) for m in s['messages'])
    print(f"   [{i:2d}] {' → '.join(roles)} | {total_chars:,} chars")

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Configura o tokenizer com o chat template correto para Mistral
tokenizer = get_chat_template(
    tokenizer,
    chat_template="mistral",  # Template Mistral Instruct
)

# Função para formatar os dados no template Mistral
def format_sample(sample):
    """Formata uma amostra para o chat template do Mistral."""
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Cria o dataset HuggingFace
dataset = Dataset.from_list(samples)
dataset = dataset.map(format_sample)

print(f"\n✅ Dataset formatado: {len(dataset)} amostras")
print(f"\n📝 Exemplo formatado (primeiros 500 chars):")
print(dataset[0]["text"][:500])

## 🚀 5. Treinar o Modelo

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ==========================================
# HIPERPARÂMETROS DE TREINO
# ==========================================
EPOCHS = 10           # Número de épocas (mais épocas = mais aprendizado, risco de overfit)
BATCH_SIZE = 2         # Batch size por dispositivo
GRAD_ACCUM = 4         # Gradient accumulation (efetivo batch = 2 × 4 = 8)
LEARNING_RATE = 2e-4   # Taxa de aprendizado
WARMUP_STEPS = 5       # Steps de warmup
MAX_STEPS = -1         # -1 = usar épocas
WEIGHT_DECAY = 0.01    # L2 regularization

print("🚀 Configuração de Treino:")
print(f"   Épocas: {EPOCHS}")
print(f"   Batch size efetivo: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Dataset size: {len(dataset)} amostras")
print(f"   Steps por época: ~{len(dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: ~{EPOCHS * len(dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,  # Empacota amostras curtas juntas para eficiência
    args=TrainingArguments(
        output_dir="/content/mother-mistral-nemo-lora",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=3,
        seed=42,
        optim="adamw_8bit",  # Otimizador 8-bit para economia de VRAM
        report_to="none",
    ),
)

In [ ]:
# ==========================================
# 🔥 INICIA O TREINO
# ==========================================
import time

print("🔥 Iniciando fine-tuning...")
print("   (Acompanhe o progresso abaixo)")
print("="*60)

start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time

print("="*60)
print(f"\n✅ Treino concluído!")
print(f"   ⏱️ Tempo total: {elapsed/60:.1f} minutos")
print(f"   📉 Loss final: {trainer_stats.training_loss:.4f}")
print(f"   📊 Steps totais: {trainer_stats.global_step}")

# Mostra VRAM usada
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3
    print(f"   💾 Peak VRAM: {peak_mem:.1f} GB")

## 🧪 6. Testar o Modelo Fine-tuned

In [ ]:
# ==========================================
# TESTE: Verifica se o modelo aprendeu
# ==========================================
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)

# Perguntas de teste
test_queries = [
    "Quem é você? O que é MOTHER?",
    "Explique os fatores de segurança para barragens de terra conforme ABNT NBR 11682.",
    "What are the key monitoring parameters for a tailings storage facility?",
]

for i, query in enumerate(test_queries):
    print(f"\n{'='*60}")
    print(f"🧪 Teste {i+1}: {query[:80]}...")
    print(f"{'='*60}")
    
    messages = [
        {"role": "system", "content": "Você é MOTHER (Multi-Objective Transformer for Human Expertise and Research), criada pela IntellTech. Sistema cognitivo autônomo com 7 camadas de processamento integrado para engenharia de barragens, geotecnia e monitoramento estrutural."},
        {"role": "user", "content": query},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(response[:1000])
    print()

## 💾 7. Salvar o Modelo

In [ ]:
# ==========================================
# SALVAR — LoRA Adapter (leve, ~50MB)
# ==========================================
LORA_OUTPUT = "/content/mother-mistral-nemo-lora-adapter"

model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)

print(f"✅ LoRA adapter salvo em: {LORA_OUTPUT}")
print(f"   Arquivos:")
for f in os.listdir(LORA_OUTPUT):
    size = os.path.getsize(os.path.join(LORA_OUTPUT, f)) / 1024
    print(f"   {f}: {size:.0f} KB")

In [ ]:
# ==========================================
# OPÇÃO A: Upload para HuggingFace Hub
# (permite usar depois via API)
# ==========================================

# Descomente e configure para fazer upload:
# HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxx"  # Seu token HuggingFace
# HF_REPO = "seu-usuario/mother-mistral-nemo-lora"  # Nome do repositório

# model.push_to_hub(HF_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
# print(f"✅ Modelo publicado em: https://huggingface.co/{HF_REPO}")

In [ ]:
# ==========================================
# OPÇÃO B: Exportar como GGUF para uso local
# (compatível com llama.cpp, Ollama, LM Studio)
# ==========================================
GGUF_OUTPUT = "/content/mother-mistral-nemo-gguf"

# Exporta em formato GGUF Q4_K_M (boa relação qualidade/tamanho)
model.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer,
    quantization_method="q4_k_m",  # Opções: q4_k_m, q5_k_m, q8_0, f16
)

print(f"\n✅ GGUF exportado em: {GGUF_OUTPUT}")
for f in os.listdir(GGUF_OUTPUT):
    size = os.path.getsize(os.path.join(GGUF_OUTPUT, f)) / 1024**2
    print(f"   {f}: {size:.0f} MB")

In [ ]:
# ==========================================
# OPÇÃO C: Download dos arquivos
# ==========================================
try:
    from google.colab import files
    import shutil
    
    # Compacta o adapter LoRA para download
    zip_path = shutil.make_archive(
        '/content/mother-mistral-nemo-lora',
        'zip',
        LORA_OUTPUT
    )
    print(f"📦 Arquivo zip: {zip_path}")
    print(f"   Tamanho: {os.path.getsize(zip_path)/1024**2:.1f} MB")
    print("\n⬇️ Iniciando download...")
    files.download(zip_path)
except ImportError:
    print("Não está no Colab. Arquivos salvos em:", LORA_OUTPUT)

## 📋 8. Resumo e Próximos Passos

### O que foi feito:
- ✅ Fine-tuned `Mistral-Nemo-Instruct-2407` com LoRA
- ✅ 12 exemplos SFT bilíngues (PT/EN) da MOTHER
- ✅ Exportado como LoRA adapter e GGUF

### Integração com MOTHER Pipeline:

**Opção 1 — HuggingFace Inference API:**
```python
# Após push_to_hub, use a Inference API do HuggingFace
import requests
API_URL = "https://api-inference.huggingface.co/models/seu-usuario/mother-mistral-nemo-lora"
headers = {"Authorization": "Bearer hf_xxxx"}
response = requests.post(API_URL, json={"inputs": "..."})
```

**Opção 2 — Ollama local:**
```bash
# Importa o GGUF no Ollama
ollama create mother-mistral -f Modelfile
ollama run mother-mistral
```

**Opção 3 — vLLM server:**
```bash
python -m vllm.entrypoints.openai.api_server \
  --model ./mother-mistral-nemo-lora-adapter \
  --port 8000
```